# 0DTE SPX Debit Spread — Exploration Notebook

Use this notebook to explore parsed data, feature distributions, model outputs, and backtest results.

**Run the pipeline first:**
```bash
python data_pipeline/parse_dbn.py
python data_pipeline/fetch_yfinance.py
python data_pipeline/merge.py
python features/engineer.py
python models/train.py
python models/evaluate.py
python backtest/engine.py
python grid_search/sweep.py
```

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.options.display.max_columns = 50
pd.options.display.float_format = '{:.4f}'.format

## 1. Load Processed Features

In [ ]:
features_path = PROJECT_ROOT / 'data' / 'processed' / 'features.parquet'
df = pd.read_parquet(features_path)
df['date'] = pd.to_datetime(df['date'])
print(f'Rows: {len(df):,}  |  Cols: {len(df.columns)}')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
df.head()

## 2. Feature Distributions

In [ ]:
from features.engineer import FEATURE_COLUMNS

feat_cols = [c for c in FEATURE_COLUMNS if c in df.columns]
fig = make_subplots(rows=4, cols=4, subplot_titles=feat_cols)

for i, col in enumerate(feat_cols):
    row = i // 4 + 1
    col_idx = i % 4 + 1
    fig.add_trace(
        go.Histogram(x=df[col].dropna(), nbinsx=40, name=col, showlegend=False),
        row=row, col=col_idx
    )

fig.update_layout(height=1000, title='Feature Distributions')
fig.show()

## 3. Label Distribution & Class Balance

In [ ]:
if 'target' in df.columns:
    label_counts = df['target'].value_counts().sort_index()
    label_names = {-1: 'DOWN (-1)', 0: 'FLAT (0)', 1: 'UP (+1)'}
    fig = px.bar(
        x=[label_names[l] for l in label_counts.index],
        y=label_counts.values,
        title='Label Distribution',
        labels={'x': 'Label', 'y': 'Count'},
        color=[label_names[l] for l in label_counts.index],
        color_discrete_map={'DOWN (-1)': 'crimson', 'FLAT (0)': 'gray', 'UP (+1)': 'steelblue'},
    )
    fig.show()
    print(label_counts)

## 4. VIX Over Time

In [ ]:
if 'vix_level' in df.columns:
    fig = px.line(df, x='date', y='vix_level', title='VIX Level Over Time')
    fig.add_hline(y=20, line_dash='dash', line_color='orange', annotation_text='VIX=20')
    fig.show()

## 5. Backtest Trade Log

In [ ]:
trade_log_path = PROJECT_ROOT / 'backtest' / 'results' / 'trade_log.parquet'
if trade_log_path.exists():
    trades = pd.read_parquet(trade_log_path)
    trades['date'] = pd.to_datetime(trades['date'])
    trades['cumulative_pnl'] = trades['pnl'].cumsum()

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=trades['date'], y=trades['cumulative_pnl'],
        mode='lines', name='Cumulative P&L',
        line=dict(color='steelblue', width=2)
    ))
    fig.update_layout(title='Cumulative P&L — Backtest (2023–present)',
                      xaxis_title='Date', yaxis_title='P&L ($)')
    fig.show()
    print(f'Total trades: {len(trades)}')
    print(f'Win rate: {trades["win"].mean():.1%}')
    print(f'Total P&L: ${trades["pnl"].sum():.2f}')
else:
    print('No trade log found. Run: python backtest/engine.py')

## 6. Load Grid Search Results

In [ ]:
sweep_path = PROJECT_ROOT / 'grid_search' / 'results' / 'sweep.parquet'
if sweep_path.exists():
    sweep = pd.read_parquet(sweep_path)
    print(f'Sweep combinations: {len(sweep):,}')
    print('\nTop 10 by Sharpe:')
    display(sweep.nlargest(10, 'sharpe')[[
        'spread_width', 'profit_target_pct', 'stop_loss_pct',
        'vix_filter', 'win_rate', 'sharpe', 'total_pnl'
    ]])
else:
    print('No sweep results found. Run: python grid_search/sweep.py')

## 7. SHAP Plots (from evaluate.py)

In [ ]:
import IPython.display as ipy

shap_path = PROJECT_ROOT / 'models' / 'shap_summary.html'
if shap_path.exists():
    ipy.display(ipy.IFrame(src=str(shap_path), width='100%', height=600))
else:
    print('Run python models/evaluate.py to generate SHAP plots.')